In [1]:
# Test ConsensusLeidenClustering
import igraph as ig
import numpy as np
import pandas as pd
from skclust.graph import ConsensusLeidenClustering, cluster_membership_cooccurrence

# ============================================================================
# Setup: Create test graph
# ============================================================================
graph = ig.Graph.Famous('Zachary')
graph.vs['name'] = [f'node_{i}' for i in range(graph.vcount())]
np.random.seed(42)
graph.es['weight'] = np.random.uniform(0.1, 1.0, graph.ecount())

print(f"Test graph: {graph.vcount()} nodes, {graph.ecount()} edges")

print("\n" + "=" * 80)
print("TEST 1: Basic functionality with verbose output")
print("=" * 80)

leiden = ConsensusLeidenClustering(n_iter=10, n_jobs=1, random_state=42, verbose=2)
leiden.fit(graph)

print(f"\nPartitions shape: {leiden.partitions_.shape}")
print(f"Membership matrix shape: {leiden.membership_matrix_.shape}")
print(f"Consensus edges: {len(leiden.consensus_edges_)} / {graph.ecount()} edges")
print(f"Consensus ratio: mean={leiden.consensus_ratio_.mean():.3f}, median={leiden.consensus_ratio_.median():.3f}")

labels = leiden.transform(graph)
print(f"\nCluster labels:\n{labels.value_counts()}")
print(f"Number of clusters: {leiden.n_clusters_}")
print(f"Excluded nodes: {len(leiden.excluded_nodes_)}")
print(f"Consensus graph: {leiden.consensus_graph_.vcount()} nodes, {leiden.consensus_graph_.ecount()} edges")
print(f"Filtered graph: {leiden.filtered_graph_.vcount()} nodes, {leiden.filtered_graph_.ecount()} edges")
print(f"\nModularity:\n{leiden.modularity_}")
print(f"\nStability report:\n{leiden.stability_report_}")

print("\n" + "=" * 80)
print("TEST 2: Resolution parameter sweep")
print("=" * 80)

for res in [0.5, 1.0, 1.5, 2.0]:
    leiden_res = ConsensusLeidenClustering(
        n_iter=10, resolution_parameter=res, n_jobs=-1, random_state=42, verbose=0
    )
    leiden_res.fit(graph)
    n_clusters_per_iter = leiden_res.partitions_.nunique(axis=0)
    print(f"Resolution {res}: "
          f"consensus edges = {len(leiden_res.consensus_edges_):3d}, "
          f"clusters = {leiden_res.n_clusters_:2d}, "
          f"avg leiden clusters = {n_clusters_per_iter.mean():.1f}, "
          f"modularity(filtered) = {leiden_res.modularity_['filtered']:.3f}")

print("\n" + "=" * 80)
print("TEST 3: Weighted graph")
print("=" * 80)

leiden_weighted = ConsensusLeidenClustering(
    n_iter=10, weight='weight', n_jobs=-1, random_state=42, verbose=0
)
labels_weighted = leiden_weighted.fit_transform(graph)

print(f"Weighted consensus edges: {len(leiden_weighted.consensus_edges_)} / {graph.ecount()}")
print(f"Weighted mean consensus: {leiden_weighted.consensus_ratio_.mean():.3f}")
print(f"Weighted clusters: {leiden_weighted.n_clusters_}")
print(f"Weighted modularity: {leiden_weighted.modularity_['filtered']:.3f}")

print("\n" + "=" * 80)
print("TEST 4: Different partition types")
print("=" * 80)

from leidenalg import ModularityVertexPartition, CPMVertexPartition

leiden_mod = ConsensusLeidenClustering(
    n_iter=10, partition_type=ModularityVertexPartition, n_jobs=-1, random_state=42, verbose=0
)
labels_mod = leiden_mod.fit_transform(graph)
print(f"ModularityVertexPartition: {len(leiden_mod.consensus_edges_)} consensus edges, {leiden_mod.n_clusters_} clusters")

leiden_cpm = ConsensusLeidenClustering(
    n_iter=10, partition_type=CPMVertexPartition, leiden_kws={'resolution_parameter': 0.1},
    n_jobs=-1, random_state=42, verbose=0
)
labels_cpm = leiden_cpm.fit_transform(graph)
print(f"CPMVertexPartition: {len(leiden_cpm.consensus_edges_)} consensus edges, {leiden_cpm.n_clusters_} clusters")

print("\n" + "=" * 80)
print("TEST 5: cluster_membership_cooccurrence standalone")
print("=" * 80)

df_partitions = leiden.partitions_
print(f"Input partitions shape: {df_partitions.shape}")

# All-pairs path
cooccur_all = cluster_membership_cooccurrence(df_partitions)
n_expected_pairs = (graph.vcount() * (graph.vcount() - 1)) // 2
assert cooccur_all.shape[0] == n_expected_pairs, f"Expected {n_expected_pairs} pairs, got {cooccur_all.shape[0]}"
print(f"✓ All-pairs: {cooccur_all.shape[0]} pairs (expected {n_expected_pairs})")

# Edge-list path (should match class output)
edge_list = [frozenset([graph.vs[e.source]['name'], graph.vs[e.target]['name']]) for e in graph.es]
cooccur_edges = cluster_membership_cooccurrence(df_partitions, edge_list=edge_list)
assert cooccur_edges.shape[0] == graph.ecount(), f"Expected {graph.ecount()} edges, got {cooccur_edges.shape[0]}"
assert cooccur_edges.equals(leiden.membership_matrix_), "Edge-list path does not match class output!"
print(f"✓ Edge-list path: {cooccur_edges.shape[0]} edges, matches class output")

print("\n" + "=" * 80)
print("TEST 6: Parallel vs Sequential consistency")
print("=" * 80)

leiden_seq = ConsensusLeidenClustering(n_iter=10, n_jobs=1, random_state=999, verbose=0)
labels_seq = leiden_seq.fit_transform(graph)

leiden_par = ConsensusLeidenClustering(n_iter=10, n_jobs=-1, random_state=999, verbose=0)
labels_par = leiden_par.fit_transform(graph)

assert leiden_seq.partitions_.equals(leiden_par.partitions_), "Sequential != Parallel partitions!"
assert leiden_seq.consensus_edges_ == leiden_par.consensus_edges_, "Sequential != Parallel consensus!"
assert labels_seq.equals(labels_par), "Sequential != Parallel labels!"
print("✓ Sequential and parallel execution produce identical results")

print("\n" + "=" * 80)
print("TEST 7: fit_transform vs fit_predict")
print("=" * 80)

leiden_ft = ConsensusLeidenClustering(n_iter=10, n_jobs=-1, random_state=42, verbose=0)
labels_ft = leiden_ft.fit_transform(graph)

leiden_fp = ConsensusLeidenClustering(n_iter=10, n_jobs=-1, random_state=42, verbose=0)
labels_fp = leiden_fp.fit_predict(graph)

assert labels_ft.equals(labels_fp), "fit_transform != fit_predict!"
print("✓ fit_transform and fit_predict produce identical results")

print("\n" + "=" * 80)
print("TEST 8: minimum_cluster_size filtering")
print("=" * 80)

leiden_min1 = ConsensusLeidenClustering(
    n_iter=10, minimum_cluster_size=1, n_jobs=-1, random_state=42, verbose=0
)
leiden_min1.fit(graph)

leiden_min5 = ConsensusLeidenClustering(
    n_iter=10, minimum_cluster_size=5, n_jobs=-1, random_state=42, verbose=0
)
leiden_min5.fit(graph)

print(f"min_size=1: {leiden_min1.n_clusters_} clusters, {len(leiden_min1.labels_)} nodes, {len(leiden_min1.excluded_nodes_)} excluded")
print(f"min_size=5: {leiden_min5.n_clusters_} clusters, {len(leiden_min5.labels_)} nodes, {len(leiden_min5.excluded_nodes_)} excluded")

# More nodes should be excluded with higher minimum
assert len(leiden_min5.excluded_nodes_) >= len(leiden_min1.excluded_nodes_), "Higher min_size should exclude more nodes"
assert leiden_min5.n_clusters_ <= leiden_min1.n_clusters_, "Higher min_size should have fewer clusters"

# Verify excluded + included = total
assert len(leiden_min5.labels_) + len(leiden_min5.excluded_nodes_) == graph.vcount(), "labels + excluded must equal total nodes"
print("✓ minimum_cluster_size filtering works correctly")

# Verify filtered_graph_ has correct node count
assert leiden_min5.filtered_graph_.vcount() == len(leiden_min5.labels_), "filtered_graph_ node count must match labels"
print("✓ filtered_graph_ node count matches labels")

# Verify modularity series
assert set(leiden_min5.modularity_.index) == {'initial', 'consensus', 'filtered'}, "modularity_ must have 3 keys"
print(f"✓ Modularity: initial={leiden_min5.modularity_['initial']:.3f}, consensus={leiden_min5.modularity_['consensus']:.3f}, filtered={leiden_min5.modularity_['filtered']:.3f}")

print("\n" + "=" * 80)
print("TEST 9: consensus_threshold")
print("=" * 80)

leiden_strict = ConsensusLeidenClustering(
    n_iter=50, consensus_threshold=1.0, n_jobs=-1, random_state=42, verbose=0
)
leiden_strict.fit(graph)

leiden_relaxed = ConsensusLeidenClustering(
    n_iter=50, consensus_threshold=0.8, n_jobs=-1, random_state=42, verbose=0
)
leiden_relaxed.fit(graph)

print(f"threshold=1.0: {len(leiden_strict.consensus_edges_)} consensus edges, {leiden_strict.n_clusters_} clusters")
print(f"threshold=0.8: {len(leiden_relaxed.consensus_edges_)} consensus edges, {leiden_relaxed.n_clusters_} clusters")

assert len(leiden_relaxed.consensus_edges_) >= len(leiden_strict.consensus_edges_), "Relaxed threshold should have more edges"
print("✓ Lower threshold retains more edges")

print("\n" + "=" * 80)
print("TEST 10: get_feature_names_out")
print("=" * 80)

edges_idx = leiden.get_feature_names_out()
assert isinstance(edges_idx, pd.Index), "get_feature_names_out must return pd.Index"
assert all(isinstance(e, frozenset) for e in edges_idx), "All entries must be frozenset"
assert len(edges_idx) == leiden.consensus_graph_.ecount(), "Length must match consensus graph edge count"
print(f"✓ get_feature_names_out: {len(edges_idx)} frozenset edges")

print("\n" + "=" * 80)
print("TEST 11: Error handling")
print("=" * 80)

leiden_error = ConsensusLeidenClustering()

try:
    leiden_error.transform(graph)
    print("✗ Should have raised RuntimeError")
except RuntimeError as e:
    print(f"✓ transform before fit: {e}")

graph_no_name = ig.Graph.Famous('Zachary')
try:
    leiden_error.fit(graph_no_name)
    print("✗ Should have raised ValueError")
except ValueError as e:
    print(f"✓ Missing 'name' attribute: {e}")

leiden_bad_weight = ConsensusLeidenClustering(weight='nonexistent')
try:
    leiden_bad_weight.fit(graph)
    print("✗ Should have raised ValueError")
except ValueError as e:
    print(f"✓ Invalid weight attribute: {e}")

try:
    ConsensusLeidenClustering(consensus_threshold=1.5).fit(graph)
    print("✗ Should have raised ValueError")
except ValueError as e:
    print(f"✓ Invalid consensus_threshold: {e}")

try:
    ConsensusLeidenClustering(minimum_cluster_size=0).fit(graph)
    print("✗ Should have raised ValueError")
except ValueError as e:
    print(f"✓ Invalid minimum_cluster_size: {e}")

print("\n" + "=" * 80)
print("TEST 12: Custom cluster prefix and labels structure")
print("=" * 80)

leiden_custom = ConsensusLeidenClustering(
    n_iter=10, cluster_prefix="community_", n_jobs=-1, random_state=42, verbose=0
)
labels_custom = leiden_custom.fit_transform(graph)

assert all(l.startswith("community_") for l in labels_custom.values), "All labels must use custom prefix"
assert labels_custom.isna().sum() == 0, "labels_ should not contain NaN"
assert labels_custom.index.name == "Node", "Index name must be 'Node'"
assert labels_custom.name == "Cluster", "Series name must be 'Cluster'"
print(f"✓ Custom prefix, no NaN, correct names:\n{labels_custom.value_counts().head()}")

# Verify excluded_nodes_ is pd.Index
assert isinstance(leiden_custom.excluded_nodes_, pd.Index), "excluded_nodes_ must be pd.Index"
assert leiden_custom.excluded_nodes_.name == "Node", "excluded_nodes_ name must be 'Node'"
print(f"✓ excluded_nodes_: pd.Index with {len(leiden_custom.excluded_nodes_)} entries")

# Verify stability_report_ is pd.Series
assert isinstance(leiden_custom.stability_report_, pd.Series), "stability_report_ must be pd.Series"
assert leiden_custom.stability_report_.name == "Stability", "stability_report_ name must be 'Stability'"
print(f"✓ stability_report_: pd.Series with name='Stability'")

print("\n" + "=" * 80)
print("ALL TESTS PASSED ✓")
print("=" * 80)

2026-02-25 08:57:36.485 | INFO     | skclust.graph:_log:368 - Validating input graph
2026-02-25 08:57:36.485 | INFO     | skclust.graph:_log:368 - Graph: 34 nodes, 78 edges
2026-02-25 08:57:36.501 | INFO     | skclust.graph:_log:368 - Using 1 parallel jobs
2026-02-25 08:57:36.501 | INFO     | skclust.graph:_log:368 - Running 10 Leiden iterations


Test graph: 34 nodes, 78 edges

TEST 1: Basic functionality with verbose output


Leiden clustering:   0%|          | 0/10 [00:00<?, ?it/s]

2026-02-25 08:57:36.517 | INFO     | skclust.graph:_log:368 - Leiden iterations completed in 0.02s
2026-02-25 08:57:36.518 | INFO     | skclust.graph:_log:368 - Computing cluster membership co-occurrence matrix
2026-02-25 08:57:36.519 | INFO     | skclust.graph:_log:368 - Co-occurrence matrix: (78, 10), computed in 0.00s
2026-02-25 08:57:36.520 | INFO     | skclust.graph:_log:368 - Found 57 consensus edges (>= 1.0 threshold)
2026-02-25 08:57:36.520 | INFO     | skclust.graph:_log:368 - Building consensus graph


Filtering consensus edges:   0%|          | 0/78 [00:00<?, ?it/s]

2026-02-25 08:57:36.523 | INFO     | skclust.graph:_log:368 - Consensus graph: 34 nodes, 57 edges (0.00s)
2026-02-25 08:57:36.524 | INFO     | skclust.graph:_log:368 - Computing connected components for cluster labels
2026-02-25 08:57:36.525 | INFO     | skclust.graph:_log:368 - Found 4 clusters in 0.00s
2026-02-25 08:57:36.525 | INFO     | skclust.graph:_log:368 - Building filtered graph and computing modularity
2026-02-25 08:57:36.525 | INFO     | skclust.graph:_log:368 - Filtered graph: 34 nodes, 78 edges
2026-02-25 08:57:36.526 | INFO     | skclust.graph:_log:368 - Modularity (initial=0.4198, consensus=0.6753, filtered=0.4198)
2026-02-25 08:57:36.526 | INFO     | skclust.graph:_log:368 - Total fit time: 0.04s
2026-02-25 08:57:36.526 | INFO     | skclust.graph:fit:638 - ============================================================
2026-02-25 08:57:36.527 | INFO     | skclust.graph:fit:639 - CONSENSUS LEIDEN CLUSTERING SUMMARY
2026-02-25 08:57:36.527 | INFO     | skclust.graph:fit:640


Partitions shape: (34, 10)
Membership matrix shape: (78, 10)
Consensus edges: 57 / 78 edges
Consensus ratio: mean=0.731, median=1.000

Cluster labels:
Cluster
leiden_1    12
leiden_2    11
leiden_3     6
leiden_4     5
Name: count, dtype: int64
Number of clusters: 4
Excluded nodes: 0
Consensus graph: 34 nodes, 57 edges
Filtered graph: 34 nodes, 78 edges

Modularity:
initial      0.419790
consensus    0.675285
filtered     0.419790
Name: Modularity, dtype: float64

Stability report:
n_edges             78.000000
mean_consensus       0.730769
median_consensus     1.000000
std_consensus        0.443560
min_consensus        0.000000
max_consensus        1.000000
pct_100             73.076923
pct_90plus          73.076923
pct_80plus          73.076923
pct_70plus          73.076923
pct_60plus          73.076923
pct_below_50        26.923077
q25                  0.000000
q75                  1.000000
Name: Stability, dtype: float64

TEST 2: Resolution parameter sweep
Resolution 0.5: consensu

In [3]:
leiden_min5.modularity_

initial      0.419790
consensus    0.675285
filtered     0.419790
Name: Modularity, dtype: float64